# Intelligent Calculator: LangChain agent, LangSmith tracing and MCP

An agent built with LangChain and Groq (`openai/gpt-oss-120b`) that picks between two exact calculator tools. The same agent is served to ChatGPT as an MCP server through an OpenAI Secure MCP Tunnel (`server.py` in this repository).

**Evidence rule for this notebook.** Every output below was saved from a real run in Colab. Where something was *not* demonstrated, it says so in plain words. The last section is a status table of what is shown and what is not.

In [ ]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

## 1. Model client
Loads the Groq key with `getpass` (never stored in the file) and creates the model with `temperature=0`, a 30 s timeout and one retry.

In [ ]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

# Retrieve current key and check if it is corrupted (e.g., contains code instead of a raw key)
api_key = os.environ.get("GROQ_API_KEY", "")
if not api_key or "import " in api_key or "ChatGroq" in api_key:
    print("No valid key found or existing key is corrupted.")
    os.environ["GROQ_API_KEY"] = getpass("Please enter your Groq API key (starts with gsk_): ").strip()

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0, timeout=30, max_retries=1)
print("Model client configured. Run the next cell to test your connection!")

No valid key found or existing key is corrupted.
Please enter your Groq API key (starts with gsk_): ··········
Model client configured. Run the next cell to test your connection!


## 2. The model alone
With no tools, the model answers the apples question by itself. It is correct (15), but this is the model's own arithmetic, not a calculation we can inspect.

In [ ]:
APPLES = "One person brings five apples and another brings ten. They put them together. How many apples are there?"
plain_reply = model.invoke([("human", APPLES)])
print(plain_reply.content)

There are **15 apples** in total.


## 3. A role changes the style, not the method
The system prompt shortens the reply to one sentence with units. The answer is still produced by the model's own reasoning, with no tool involved.

In [ ]:
role_reply = model.invoke([
    ("system", "You are a patient math tutor. Give one short sentence with the answer and its units."),
    ("human", APPLES),
])
print(role_reply.content)

There are 15 apples.


## 4. The `add` tool
An ordinary Python function turned into a tool. The model reads its name and docstring to decide when to use it.

In [ ]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.name)

add


## 5. The `tree_distance` tool
Horizontal distance is `height / tan(angle)`. Bad input returns a message instead of raising an error, so the agent can ask for a correction.

In [ ]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.name)

tree_distance


## 6. Testing the tools directly
**What the output shows:** `add(5, 10)` returns 15. `tree_distance(10 m, 30°)` returns 17.32 m, which matches 10 / tan 30° = 17.3205. An angle of 0° is rejected with a clear message. The tools are correct before any model is involved.

In [ ]:
print("Apples:", add.invoke({"a": 5, "b": 10}))
print("Tree distance in metres:", tree_distance.invoke({"height_m": 10, "angle_deg": 30}))
print("Invalid angle:", tree_distance.invoke({"height_m": 10, "angle_deg": 0}))

Apples: 15
Tree distance in metres: 17.32
Invalid angle: Invalid angle. Supply an angle strictly between 0 and 90 degrees.


## 7. The agent
The agent gets both tools plus the instructions: call the matching tool once, ask for missing values, never guess, and ask for a correction on invalid input.

In [ ]:
from langchain.agents import create_agent

INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)
print("Agent ready")

Agent ready


## 8. The run viewer
`show_run` prints every tool request, every tool result, the number of tool requests, and the final answer, so each run's behaviour is visible and checkable.

In [ ]:
from langchain_core.messages import ToolMessage

def solve(problem, *, run_name="intro_calculator"):
    return agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": run_name},
    )

def show_run(problem):
    result = solve(problem)
    call_count = 0
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            call_count += 1
            print("Tool requested:", call["name"], call["args"])
        if isinstance(message, ToolMessage):
            print("Tool returned:", message.name, message.content)
    print("Tool requests:", call_count)
    print("Final answer:", result["messages"][-1].content)
    return result

print("Viewer ready")

Viewer ready


## 9. Saved run: apples (5 + 10)
**What the run showed:** one tool request, `add` with `a=5, b=10`, returned `15`, and the final answer was 15 apples. The count of 1 request confirms the agent used the tool once and used its result. This is the difference from section 2, where the model answered with no tool.

In [ ]:
apples_run = show_run(APPLES)

Tool requested: add {'a': 5, 'b': 10}
Tool returned: add 15
Tool requests: 1
Final answer: The total number of apples is **15 apples**.


## 10. Saved run: tree distance
**What the run showed:** one tool request, `tree_distance` with `height_m=10, angle_deg=30`, returned `17.32`, and the final answer was 17.32 m. It matches the direct check in section 6, so the agent passed the problem's numbers to the tool correctly.

In [ ]:
TREE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "The angle of elevation to the top is 30 degrees. "
    "How far away is the tree horizontally?"
)
tree_run = show_run(TREE)

Tool requested: tree_distance {'angle_deg': 30, 'height_m': 10}
Tool returned: tree_distance 17.32
Tool requests: 1
Final answer: The horizontal distance to the tree is **17.32 m**.


## 11. Saved run: missing angle
**What the run showed:** **zero** tool requests. The height was given but the angle was not, and the agent asked for the angle of elevation instead of guessing one. This is the "never guess inputs" rule working.

In [ ]:
MISSING_ANGLE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "How far away is the tree horizontally?"
)
missing_run = show_run(MISSING_ANGLE)

Tool requests: 0
Final answer: To find the horizontal distance, I need the angle of elevation from your eye level to the top of the tree. Could you tell me that angle (in degrees)?


## 12. Fresh run: 7 + 8
A new problem that none of the earlier runs used. **Status: not yet run.** Run this cell and save the notebook so its output is stored.
**What to check in the output:** one `Tool requested: add` line with `a=7, b=8` (not `5` and `10`), `Tool returned: add 15`, and `Tool requests: 1`. Both problems add up to 15, so check the *arguments*, not just the answer, to confirm this is a new run.

In [ ]:
FRESH = "One person brings seven apples and another brings eight. They put them together. How many apples are there?"
fresh_run = show_run(FRESH)

## 13. LangSmith tracing: first attempt (failed)
The next two cells configure a LangSmith client and run three traced calls. **The saved output below shows the attempt did not work:** LangSmith rejected the agent id `intro build calculator` because it contains spaces. Agent ids may only use lowercase letters, digits and hyphens, and must start with a letter. The warning also states that calls with no valid destination are not traced.

So the line "Look for intro_apples, intro_tree and intro_missing_angle" printed at the end of cell 13 does **not** mean traces exist. **LangSmith tracing is not demonstrated by these two cells.**

In [ ]:
from langsmith import Client, tracing_context

destination = input("LangSmith destination type (agent/project): ").strip().lower()
if destination == "agent":
    os.environ.pop("LANGSMITH_PROJECT", None)
    os.environ["LANGSMITH_AGENT_ID"] = input("Agent ID: ").strip()
    os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = input(
        "Environment (for example local): "
    ).strip()
elif destination == "project":
    os.environ.pop("LANGSMITH_AGENT_ID", None)
    os.environ.pop("LANGSMITH_AGENT_ENVIRONMENT", None)
    os.environ["LANGSMITH_PROJECT"] = input("Project name: ").strip()
else:
    raise ValueError("Enter agent or project, then rerun this cell.")

endpoint = input("LangSmith API URL (blank for default US): ").strip()
if endpoint:
    os.environ["LANGSMITH_ENDPOINT"] = endpoint
workspace_id = input("Workspace ID (blank if unnecessary): ").strip()
if workspace_id:
    os.environ["LANGSMITH_WORKSPACE_ID"] = workspace_id
smith_key = getpass("LangSmith API key: ").strip()
smith_client = Client(api_key=smith_key)
print("Tracing client configured.")

LangSmith destination type (agent/project): agent
Agent ID: intro build calculator
Environment (for example local): local
LangSmith API URL (blank for default US): 
Workspace ID (blank if unnecessary): 
LangSmith API key: ··········
Tracing client configured.


/tmp/ipykernel_708/3819367807.py:24: LangSmithWarning: The LANGSMITH_AGENT_* env vars can't address a run (LANGSMITH_AGENT_ID='intro build calculator', LANGSMITH_AGENT_ENVIRONMENT='local'): Address agent id must be 1 to 63 lowercase ASCII letters, digits, or hyphens, start with a letter, and end with a letter or digit, got 'intro build calculator'. Use an id such as 'support-agent'. Calls that name no destination in code are not traced.
  smith_client = Client(api_key=smith_key)


In [ ]:
with tracing_context(enabled=True, client=smith_client):
    solve(APPLES, run_name="intro_apples")
    solve(TREE, run_name="intro_tree")
    solve(MISSING_ANGLE, run_name="intro_missing_angle")
smith_client.flush()
print("Look for intro_apples, intro_tree and intro_missing_angle.")

Look for intro_apples, intro_tree and intro_missing_angle.


## 14. LangSmith tracing: corrected rerun
**Status: not yet run.** This cell reuses the `smith_client` from section 13, sets a valid agent id (`intro-build-calculator`), and checks the id format before tracing so the same mistake fails loudly instead of silently. Run it, then open LangSmith and look for the runs `intro_apples_v2`, `intro_tree_v2` and `intro_missing_angle_v2`. Paste the trace link into the status table below.

In [ ]:
import re
agent_id = "intro-build-calculator"
assert re.fullmatch(r"[a-z]([a-z0-9-]{0,61}[a-z0-9])?", agent_id), "Invalid LangSmith agent id"
os.environ["LANGSMITH_AGENT_ID"] = agent_id
os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = "local"

with tracing_context(enabled=True, client=smith_client):
    solve(APPLES, run_name="intro_apples_v2")
    solve(TREE, run_name="intro_tree_v2")
    solve(MISSING_ANGLE, run_name="intro_missing_angle_v2")
smith_client.flush()
print("Agent id used:", agent_id)
print("Now open LangSmith and look for the three *_v2 runs. No warning above means tracing was accepted.")

## 15. MCP server and Secure MCP Tunnel
The same agent is exposed to ChatGPT as one tool, `solve_math_problem`, by `server.py` in this repository (FastMCP, `stateless_http=True`, read-only annotation, input length check). It listens on `http://127.0.0.1:8000/mcp`. ChatGPT reaches it through an OpenAI Secure MCP Tunnel and `tunnel-client`, configured with the no-auth sample profile and **No authentication** in the ChatGPT plugin form. The server needs `mcp<2`, because version 2 renamed `FastMCP`.

**Evidence that exists:** the tunnel-client dashboard showed *Health: live*, *Ready: ready* and the `main` channel enabled over `http-streamable`.
**Evidence that does not exist yet in this repository:** a saved ChatGPT conversation calling `solve_math_problem` through the tunnel. Add a screenshot or transcript of `@Intelligent Calculator` answering the apples prompt to demonstrate the full path.

## 16. Verification status

| Check | Status | Evidence |
|---|---|---|
| Tools are correct on their own | **Demonstrated** | Section 6 output: 15, 17.32, invalid-angle message |
| Agent calls `add` for apples | **Demonstrated** | Section 9: one request, `add(5, 10)` returns 15 |
| Agent calls `tree_distance` | **Demonstrated** | Section 10: one request, returns 17.32 |
| Agent asks instead of guessing | **Demonstrated** | Section 11: zero tool requests, asks for the angle |
| Fresh 7 + 8 run | **Not demonstrated yet** | Section 12 has no saved output; run it |
| LangSmith tracing | **Not demonstrated.** First attempt failed on an invalid agent id (section 13). | Run section 14 and add a trace link |
| MCP server and tunnel up | **Demonstrated** | Dashboard: Health live, Ready ready, `main` channel enabled |
| ChatGPT calls the tool through the tunnel | **Not demonstrated yet** | No transcript saved; add one |

Nothing is marked *Demonstrated* unless its output or evidence is saved in this repository.